# Get a user by number

## What it does

Shows the user with that number.

## What happens to the saved users

It creates a practice user, reads it, and deletes it at the end. The real users stay the same.

## Feature it uses

`getUser(id)`, from `logic/get-user.js`.

## Failures it can show

- A number nobody has → `User not found`.
- The user cannot be read → `Could not find the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { getUser } = await import("./logic/get-user.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  function printUsers(users) {
    const table = new Table({
      head: ["id", "name", "email", "createdAt", "updatedAt"],
      style: { head: ["cyan", "bold"] },
    });
    for (const user of users) {
      table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
    }
    console.log(table.toString());
  }

  const practice = createUser("Practice", `practice-${Date.now()}@x.com`);
  try {
    // 1. Get the user by id: a table with the practice user
    printUsers([getUser(practice.id)]);

    // 2. An id nobody has: "User not found"
    try {
      getUser(999999);
    } catch (error) {
      console.log(error.message);
    }
  } finally {
    deleteUser(practice.id);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌────┬──────────┬──────────────────────────────┬─────────────────────┬─────────────────────┐
│ id │ name     │ email                        │ createdAt           │ updatedAt           │
├────┼──────────┼──────────────────────────────┼─────────────────────┼─────────────────────┤
│ 22 │ Practice │ practice-1793027114762@x.com │ 2026-10-26 16:05:14 │ 2026-10-26 16:05:14 │
└────┴──────────┴──────────────────────────────┴─────────────────────┴─────────────────────┘
User not found
